In [220]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import os

In [221]:
path = "./raw_data.csv"

In [222]:
df = pd.read_csv(path)

In [223]:
df.columns = (
    df.columns
      .str.strip()
)

if "umber pets" in df.columns:
    df.rename(columns={"umber pets": "Number Pets"}, inplace=True)


In [224]:
df.head()

,CustomerID,Region,Gender,Age,Education Years,Employment Years,Job Category,Retired,Marital Status,Household Size,...,Coupon Redemption,Brand Tenure Months,Monthly Spend ProductA,Cumulative Spend ProductA,Monthly Spend ProductB,Cumulative Spend ProductB,Monthly Spend ProductC,Cumulative Spend ProductC,Total Avg Monthly Spend,High Value Customer
0,0002-GTOKLU-YVY,5,Male,63,16,3,Sales,No,Married,2,...,0,21,$20.60,$84.92,$147.60,$602.92,$176.50,$790.02,$344.70,1
1,0003-RLTRGE-IW2,1,Male,52,11,14,Professional,No,Unmarried,1,...,0,31,$46.20,$304.28,$0.00,$0.00,$107.25,$746.28,$153.45,0
2,0003-UTGKPR-PRU,2,Male,73,14,11,Professional,No,Unmarried,1,...,0,54,$65.20,$703.88,$0.00,$0.00,$0.00,$0.00,$65.20,0
3,0008-ZIQQOT-SGB,3,Female,40,21,3,Sales,No,Married,3,...,0,1,$8.60,$1.72,$171.20,$34.24,$150.25,$36.06,$330.05,1
4,0012-CIVYLF-839,4,Male,40,12,9,Labor,No,Married,2,...,0,48,$46.60,$495.20,$0.00,$0.00,$0.00,$0.00,$46.60,0


In [225]:

text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    if col in ["CustomerID"]:
        #don't modify the value for CustomerID
        continue
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )



In [226]:
df.columns

Index(['CustomerID', 'Region', 'Gender', 'Age', 'Education Years',
       'Employment Years', 'Job Category', 'Retired', 'Marital Status',
       'Household Size', 'Household Income', 'Number Pets', 'Home Owner',
       'Car Ownership', 'Car Brand', 'Car Value', 'Commute Distance',
       'Political Party', 'Votes', 'Credit Card', 'Credit Card Tenure',
       'Active Lifestyle', 'TV Watching Hours', 'Streaming Svcs',
       'Wireless Internet', 'Smart Phone', 'Twitter Acct', 'LinkedIn Acct',
       'Facebook Acct', 'News Subscriber', 'Coupon Redemption',
       'Brand Tenure Months', 'Monthly Spend ProductA',
       'Cumulative Spend ProductA', 'Monthly Spend ProductB',
       'Cumulative Spend ProductB', 'Monthly Spend ProductC',
       'Cumulative Spend ProductC', 'Total Avg Monthly Spend',
       'High Value Customer'],
      dtype='object')

In [227]:
binary_map = {
    "Yes": 1, "No": 0,
    "Y": 1, "N": 0,
    "True": 1, "False": 0,
    "1": 1, "0": 0
}

binary_cols = [
    "Retired",
    "Home Owner",
    "Active Lifestyle",
    "Wireless Internet",
    "Smart Phone",
    "Twitter Acct",
    "LinkedIn Acct",
    "Facebook Acct",
    "News Subscriber",
    "Coupon Redemption",
    "High Value Customer",
    "Streaming Svcs",
    "Votes",
    "Political Party"
]

for col in binary_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .map(binary_map)
        )


In [228]:
currency_cols = [
    "Household Income",
    "Car Value",
    "Monthly Spend ProductA",
    "Cumulative Spend ProductA",
    "Monthly Spend ProductB",
    "Cumulative Spend ProductB",
    "Monthly Spend ProductC",
    "Cumulative Spend ProductC",
    "Total Avg Monthly Spend"
]

for col in currency_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(r"[$,]", "", regex=True)
        )

numeric_cols = currency_cols + [
    "Age",
    "Education Years",
    "Employment Years",
    "Household Size",
    "Number Pets",
    "Commute Distance"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [229]:

# Start with all rows assumed valid and an empty list of reasons per row
invalid_mask = pd.Series(False, index=df.index)
invalid_reasons = pd.Series([[] for _ in range(len(df))], index=df.index)

# Pattern: for each condition, build a boolean mask `cond`, OR it into `invalid_mask`,
# then append a human-readable reason to `invalid_reasons` for the same rows.

# 1) Nulls in critical fields -> mark invalid
cond = df.isna().any(axis=1)
invalid_mask |= cond
invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Null value'])

# 2) Ensure numeric columns are numeric (coercion created NaN for bad values) -> mark those as invalid
if 'numeric_cols' in locals():
    cond = df[numeric_cols].isna().any(axis=1)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Invalid numeric value'])

# 3) Negative values for Education and Employment years are invalid
if 'Education Years' in df.columns:
    cond = df['Education Years'] < 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Negative Education Years'])
if 'Employment Years' in df.columns:
    cond = df['Employment Years'] < 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Negative Employment Years'])

# 4) Income must be >= 0
if 'Household Income' in df.columns:
    cond = df['Household Income'] < 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Negative Household Income'])

# 5) Household size must be > 0
if 'Household Size' in df.columns:
    cond = df['Household Size'] <= 0
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Non-positive Household Size'])

# 6) Education years must be less than Age - 5 (assuming people start school around age 5)
if set(['Education Years','Age']).issubset(df.columns):
    cond = ~(df['Education Years'] < df['Age'] - 5)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Education Years >= Age - 5'])

# 6) Employment years cannot exceed plausible range: Employment Years <= Age - 14
if set(['Employment Years','Age']).issubset(df.columns):
    cond = ~(df['Employment Years'] <= (df['Age'] - 14))
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Employment Years implausible'])

# X) Credit card consistency: Credit Card == 0 but Credit Card Tenure > 0
if set(['Credit Card','Credit Card Tenure']).issubset(df.columns):
    cc = pd.to_numeric(df['Credit Card'], errors='coerce')
    cc_tenure = pd.to_numeric(df['Credit Card Tenure'], errors='coerce')
    cond = (cc == 0) & (cc_tenure > 0)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Credit Card zero with positive tenure'])

# 7) Car value threshold: if provided and below 1000, mark invalid
if 'Car Value' in df.columns:
    # Coerce to numeric here to ensure values like 'Nocar' become NaN
    car_val = pd.to_numeric(df['Car Value'], errors='coerce')
    # Treat 0 as 'no car' (do not flag). Only flag positive values below threshold.
    cond = car_val.notna() & (car_val > 0) & (car_val < 1000)
    invalid_mask |= cond
    invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + ['Car Value below threshold (non-zero)'])

# 9) Cumulative spend checks for Products A, B, C: cumulative >= monthly
for p in ['A','B','C']:
    monthly = f'Monthly Spend Product{p}'
    cumulative = f'Cumulative_Spend_Product{p}'
    if (monthly in df.columns) and (cumulative in df.columns):
        cond = df[cumulative].isna() | df[monthly].isna() | (df[cumulative] < df[monthly])
        invalid_mask |= cond
        invalid_reasons.loc[cond] = invalid_reasons.loc[cond].apply(lambda lst: lst + [f'Cumulative < Monthly for Product{p}'])

# 10) Remove duplicate customer records: keep first, mark subsequent duplicates as invalid
if 'CustomerID' in df.columns:
    dup_mask = df.duplicated(subset='CustomerID', keep='first')
    invalid_mask |= dup_mask
    invalid_reasons.loc[dup_mask] = invalid_reasons.loc[dup_mask].apply(lambda lst: lst + ['Duplicate CustomerID'])

# Build invalid and cleaned DataFrames
invalid_df = df[invalid_mask].copy()
# Attach a single string column summarizing reasons; empty string if none
invalid_df['Invalid Reasons'] = invalid_reasons[invalid_mask].apply(lambda lst: '; '.join(lst) if lst else '')
cleaned_df = df[~invalid_mask].copy()

# Ensure cleaned_df has no duplicates (keep first) and drop them if present
if 'CustomerID' in cleaned_df.columns:
    cleaned_df = cleaned_df.drop_duplicates(subset='CustomerID', keep='first')


In [230]:
cleaned_df.head()

,CustomerID,Region,Gender,Age,Education Years,Employment Years,Job Category,Retired,Marital Status,Household Size,...,Coupon Redemption,Brand Tenure Months,Monthly Spend ProductA,Cumulative Spend ProductA,Monthly Spend ProductB,Cumulative Spend ProductB,Monthly Spend ProductC,Cumulative Spend ProductC,Total Avg Monthly Spend,High Value Customer
0,0002-GTOKLU-YVY,5,Male,63,16,3,Sales,0,Married,2,...,0,21,20.6,84.92,147.6,602.92,176.50,790.02,344.70,1
1,0003-RLTRGE-IW2,1,Male,52,11,14,Professional,0,Unmarried,1,...,0,31,46.2,304.28,0.0,0.00,107.25,746.28,153.45,0
2,0003-UTGKPR-PRU,2,Male,73,14,11,Professional,0,Unmarried,1,...,0,54,65.2,703.88,0.0,0.00,0.00,0.00,65.20,0
3,0008-ZIQQOT-SGB,3,Female,40,21,3,Sales,0,Married,3,...,0,1,8.6,1.72,171.2,34.24,150.25,36.06,330.05,1
4,0012-CIVYLF-839,4,Male,40,12,9,Labor,0,Married,2,...,0,48,46.6,495.20,0.0,0.00,0.00,0.00,46.60,0


In [231]:
invalid_df.head()

,CustomerID,Region,Gender,Age,Education Years,Employment Years,Job Category,Retired,Marital Status,Household Size,...,Brand Tenure Months,Monthly Spend ProductA,Cumulative Spend ProductA,Monthly Spend ProductB,Cumulative Spend ProductB,Monthly Spend ProductC,Cumulative Spend ProductC,Total Avg Monthly Spend,High Value Customer,Invalid Reasons
8,0016-XBFXXW-2G0,4,Female,19,14,0,Professional,0,Married,3,...,22,22.6,101.48,107.8,435.16,0.00,0.00,130.40,0,Education Years >= Age - 5
15,0024-UQUVXG-79W,1,Male,26,21,0,Sales,0,Married,3,...,9,20.6,47.04,252.2,379.80,399.75,790.08,672.55,1,Education Years >= Age - 5
60,0078-GEDTVZ-ZZ6,4,Female,23,18,0,Professional,0,Unmarried,1,...,9,36.6,68.68,0.0,0.00,0.00,0.00,36.60,0,Education Years >= Age - 5
62,0086-VMUGHD-VQ4,4,Male,22,17,0,Professional,0,Unmarried,3,...,10,10.2,27.52,0.0,0.00,0.00,0.00,10.20,0,Education Years >= Age - 5
73,0109-POJOQC-B1R,3,Female,25,20,0,Sales,0,Unmarried,1,...,5,18.0,19.44,160.8,119.96,115.00,94.86,293.80,1,Education Years >= Age - 5


In [232]:
# Create timestamped output directory: outputs/<yy-mm-dd-hh>/
ts = datetime.now().strftime('%y-%m-%d-%H')
outdir = Path('outputs') / ts
outdir.mkdir(parents=True, exist_ok=True)

In [233]:
# Save outputs as Excel workbooks
cleaned_df.to_excel(outdir / 'customer_data_cleaned.xlsx', index=False)
invalid_df.to_excel(outdir / 'invalid.xlsx', index=False)

print('Saved', cleaned_df.shape[0], 'clean rows and', invalid_df.shape[0], 'invalid rows to', outdir)

Saved 4667 clean rows and 333 invalid rows to outputs\26-04-14-16
